# Cavendish Data Analysis (Python)
Replicates the C++ workflow from `FileLab.cpp`: loads tracked points, computes angles, fits a damped oscillation, and plots residuals.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
def load_xy(path):
    arr = np.loadtxt(path)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 2)
    return arr[:, 0], arr[:, 1]

def averaged_every_n(values, n):
    trimmed = values[: len(values) - len(values) % n]
    return trimmed.reshape(-1, n).mean(axis=1)

def damped_cos(t, p0, p1, p2, p3, p4):
    return p0 + p1 * np.exp(-p3 * t) * np.cos(p2 * t + p4)

In [ ]:
points_path = Path('points2.txt')  # output from tracking
x_fix, y_fix = 50.94, 25.80          # fixed marker in pixels (from C++ config)
L = 4.574 * 100                      # mm -> assume cm scale; keep consistent with inputs
L_err = 0.004 * 100
time_video = 220 * 8                 # total duration in seconds
pixel_sigma = 0.01                   # pixel uncertainty used in C++ code

x_px, y_px = load_xy(points_path)
n = len(x_px)
t = np.linspace(0, time_video, n, endpoint=False)
t_err = np.full_like(t, 0.1)

r = np.hypot(x_fix - x_px, y_fix - y_px)
r_err = np.sqrt(2) * np.hypot(pixel_sigma, pixel_sigma)
theta = 0.5 * np.arctan(r / L)
theta_err = 0.5 * (1 / (1 + (r / L) ** 2)) * np.sqrt((r_err / L) ** 2 + (r * L_err / L**2) ** 2)

print(f'Loaded {n} points from {points_path}')

## Raw data

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(t, theta, yerr=theta_err, fmt='k.', markersize=2, elinewidth=0.5)
ax.set_xlabel('t (s)')
ax.set_ylabel('theta (rad)')
ax.set_title('Angular displacement vs time')
ax.grid(True)

## Average every 10 samples (noise reduction)

In [ ]:
block = 10
t_avg = averaged_every_n(t, block)
theta_avg = averaged_every_n(theta, block)
t_err_avg = np.sqrt(block) * averaged_every_n(t_err, block)
theta_err_avg = np.sqrt(block) * averaged_every_n(theta_err, block)

fig, ax = plt.subplots()
ax.errorbar(t_avg, theta_avg, yerr=theta_err_avg, fmt='k.', markersize=3, elinewidth=0.7)
ax.set_xlabel('t (s)')
ax.set_ylabel('theta (rad)')
ax.set_title('Averaged (10-sample blocks)')
ax.grid(True)

## Fit damped oscillation

In [ ]:
p0_guess = [0.01, -0.005, 0.012, 0.4, 0.0]
popt, pcov = curve_fit(damped_cos, t_avg, theta_avg, p0=p0_guess, sigma=theta_err_avg, absolute_sigma=True, maxfev=20000)
perr = np.sqrt(np.diag(pcov))

for i, (p, e) in enumerate(zip(popt, perr)):
    print(f'p{i}: {p:.6e} ± {e:.6e}')

t_dense = np.linspace(t_avg.min(), t_avg.max(), 2000)
fit_curve = damped_cos(t_dense, *popt)

fig, ax = plt.subplots()
ax.errorbar(t_avg, theta_avg, yerr=theta_err_avg, fmt='k.', markersize=3, elinewidth=0.7, label='data (avg)')
ax.plot(t_dense, fit_curve, 'r-', label='fit')
ax.set_xlabel('t (s)')
ax.set_ylabel('theta (rad)')
ax.legend()
ax.grid(True)

## Residuals

In [ ]:
res = theta_avg - damped_cos(t_avg, *popt)
fig, ax = plt.subplots()
ax.axhline(0, color='red', linewidth=1)
ax.plot(t_avg, res, 'k.', markersize=3)
ax.set_xlabel('t (s)')
ax.set_ylabel('residual (rad)')
ax.set_title('Fit residuals')
ax.grid(True)

## Export results

In [ ]:
out_params = Path('fit_parameters.json')
payload = {f'p{i}': {'value': float(v), 'sigma': float(e)} for i, (v, e) in enumerate(zip(popt, perr))}
out_params.write_text(json.dumps(payload, indent=2))
print('Saved fit parameters to', out_params)